# 02 · Skeleton visualisation

Front (X-Y) and side (Z-Y) views of raw sequences, and the effect of the preprocessing pipeline.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from skeleton_action.constants import CLASS_NAMES, JOINT_NAMES, KARD_EDGES, J
from skeleton_action.data.kard import KardCollection
INTERIM = ROOT / "data/interim/kard_realworld.npz"
assert INTERIM.exists(), "run `make data` first"
col = KardCollection.load(INTERIM)
print(len(col), "sequences")

In [ ]:
from skeleton_action.preprocessing.transforms import PreprocessConfig, normalize_spatial, preprocess_sequence

def draw(ax, frame, plane=(0, 1), color="k", alpha=1.0):
    for a, b in KARD_EDGES:
        ax.plot(frame[[a, b], plane[0]], frame[[a, b], plane[1]], color=color, alpha=alpha, lw=2)
    ax.scatter(frame[:, plane[0]], frame[:, plane[1]], s=12, color=color, alpha=alpha)

def show(seq_id, n=6):
    s = col.sequence(col.index_of(seq_id))
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for k, t in enumerate(np.linspace(0, len(s) - 1, n).astype(int)):
        a = 0.25 + 0.75 * k / (n - 1)
        draw(axes[0], s[t], (0, 1), plt.cm.viridis(k / n), a)
        draw(axes[1], s[t], (2, 1), plt.cm.viridis(k / n), a)
    axes[0].set(title=f"{seq_id} {CLASS_NAMES[int(seq_id[1:3]) - 1]}: front (x, y)", aspect="equal")
    axes[1].set(title="side (z, y)", aspect="equal")
    plt.tight_layout(); plt.show()

for sid in ["a02_s01_e01", "a10_s03_e01", "a14_s05_e01", "a17_s02_e01", "a18_s02_e01"]:
    show(sid)

## Joint trajectories over time

In [ ]:
s = col.sequence(col.index_of("a17_s02_e01"))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for i, axn in enumerate("xyz"):
    for j in (J.HEAD, J.TORSO, J.R_HAND, J.R_KNEE, J.R_FOOT):
        ax[i].plot(s[:, j, i], label=JOINT_NAMES[j])
    ax[i].set_title(f"Sit down - {axn} (mm)")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## Preprocessing: first-frame vs per-frame torso centring

Per-frame centring removes the vertical torso trajectory that separates *Sit down* from *Stand up*.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for sid in ["a17_s02_e01", "a18_s02_e01"]:
    raw = col.sequence(col.index_of(sid))
    for k, mode in enumerate(["first_frame", "per_frame"]):
        x = preprocess_sequence(raw, PreprocessConfig(center=mode))
        ax[k].plot(x[:, J.HEAD, 1], label=f"{sid} head y")
        ax[k].set_title(f"center={mode}: head height (body units)")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Velocity channels

In [ ]:
x = preprocess_sequence(col.sequence(col.index_of("a09_s01_e01")), PreprocessConfig(velocity=True))
print(x.shape)
plt.figure(figsize=(8, 3)); plt.plot(np.linalg.norm(x[:, :, 3:], axis=-1)[:, [J.R_FOOT, J.L_FOOT, J.R_HAND]])
plt.legend(["R foot", "L foot", "R hand"]); plt.title("Forward kick: |velocity| per resampled frame"); plt.show()